# Round 16: service-rating aggregates + seed-averaged forest - S6E10

Two bounded experiments, one change each, under the standing steer (feature engineering for all
models; random-forest preference honored with measured tests):

- **Exp 40: per-passenger service summary.** The service-rating columns enter the Exp 36
  scheme individually (raw + value-level TE). A row-level summary - mean rating and count of
  low ratings (<= 2) - is information the column-wise scheme does not express directly. The two
  aggregates are added as RAW numeric features (not counted, not target-encoded); everything
  else is the Exp 36 scheme unchanged.
- **Exp 41: seed-averaged RandomForest.** Every forest conclusion since round 13 carries the
  caveat that forests ran single-seed against an averaged booster. Exp 41 measures the
  averaging gain for the forest family itself: Exp 37 exactly, at seeds 42/43/44, OOF averaged
  per row. Booster-lineage averaging gains (0.1-0.3e-3) sit an order below the measured RF gap
  (24e-4), so expected value is low. The measurement is bounded to this config and these three
  seeds: it sizes this one averaging effect and cannot exclude every form of averaging
  asymmetry.

Guard: refit fires only on a WORKING SIGNAL win vs the Exp 36 bar. 0.960 status: the preferred
candidate is Exp 36 at 0.959996 (strictly unmet); round-15's exploratory blend numbers that
numerically crossed 0.960 were adaptive scan outputs, not candidate scores.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CROSS_PAIRS = [('Class', 'Type of Travel'), ('Customer Type', 'Class')]
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100
RF_PARAMS = dict(n_estimators=400, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE)
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, round 9
E36_FOLDS_R12 = [0.96016, 0.95887, 0.96067, 0.96009, 0.96019]       # the bar (seed-42 partition)
E37_FOLDS_R13 = [0.95756, 0.95691, 0.95824, 0.95789, 0.95754]       # single-seed RF, round 13

In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def cross_name(pair):
    return ' x '.join(pair)

CROSS_COLS = [cross_name(p) for p in CROSS_PAIRS]

def add_crosses(df):
    out = df.copy()
    for pair in CROSS_PAIRS:
        out[cross_name(pair)] = out[pair[0]].astype(str) + ' x ' + out[pair[1]].astype(str)
    return out

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False,
             target_encode=False, te_m=TE_SMOOTH, te_cols=None, te_drop=(),
             outer_seed=RANDOM_STATE, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)  # round 14: partition passed in
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 audit fix)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            cols = te_cols if te_cols is not None else TE_COLS_ALL
            te_fit = te_crossfit(X_fit, y_fit, cols, m=te_m)
            te_es = te_apply(X_fit, y_fit, X_es, cols, m=te_m)
            te_va = te_apply(X_tr, y_tr, X_va, cols, m=te_m)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
            if te_drop:  # raw crossed strings never enter the model - only their encodings
                X_fit = X_fit.drop(columns=list(te_drop))
                X_es = X_es.drop(columns=list(te_drop))
                X_va = X_va.drop(columns=list(te_drop))
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label, outer_seed=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; crossed columns: {CROSS_COLS}')
def forest_prep(X_tr, y_tr, X_va):
    # counts + leak-free TE, ordinal natives, fold-local median impute (round-13 construction)
    X_tr, (X_va,) = add_counts(X_tr, [X_va])
    te_tr = te_crossfit(X_tr, y_tr, TE_COLS_ALL)
    te_va = te_apply(X_tr, y_tr, X_va, TE_COLS_ALL)
    X_tr = pd.concat([X_tr.reset_index(drop=True), te_tr.reset_index(drop=True)], axis=1)
    X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
    for col in CATEGORICAL:
        cats = pd.Categorical(X_tr[col])
        X_tr[col] = cats.codes
        X_va[col] = pd.Categorical(X_va[col], categories=cats.categories).codes
    med = X_tr.median(numeric_only=True)
    return X_tr.fillna(med), X_va.fillna(med)

def oof_forest(feat_frame, label, name, kind, seed=RANDOM_STATE):
    oof = np.zeros(len(feat_frame))
    aucs = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        X_tr, X_va = forest_prep(X_tr, y_tr, X_va)
        cls = RandomForestClassifier if kind == 'rf' else ExtraTreesClassifier
        params = {**RF_PARAMS, 'random_state': seed}
        model = cls(**params)
        model.fit(X_tr, y_tr)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a)
        print(f'  {name} fold {fold}: AUC {a:.5f}')
    return oof, aucs

def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label


Full train: 699,635 rows, 21 raw features; crossed columns: ['Class x Type of Travel', 'Customer Type x Class']


## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every seed-42-partition round.

In [3]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 1: AUC 0.95851 (best_iteration=296)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 2: AUC 0.95751 (best_iteration=265)


...

  INC fold 3: AUC 0.95907 (best_iteration=212)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 4: AUC 0.95841 (best_iteration=183)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 5: AUC 0.95885 (best_iteration=555)
  determinism vs round-3 constants: max abs diff 0.00000 (PASS - identical folds)


## Exp 40: Exp 36 scheme + raw service-rating aggregates

The rating columns (13 in this dataset) are the numeric columns that are not Age, Flight Distance, or the two
delay columns (derived by exclusion, asserted). Aggregates: row mean and count of ratings <= 2.
Counts and TE remain on the original columns only - the aggregates enter as raw numerics.
6-model average (3 seeds x {63, 127} leaves), same as Exp 36.

In [4]:
DELAY_COLS = [c for c in features.columns if 'Delay' in c]
NON_RATING = list(CATEGORICAL) + ['Age', 'Flight Distance'] + DELAY_COLS
RATING_COLS = [c for c in features.columns if c not in NON_RATING]
assert 10 <= len(RATING_COLS) <= 15, f'unexpected rating-column count {len(RATING_COLS)}: {RATING_COLS}'
print(f'{len(RATING_COLS)} rating columns identified by exclusion: {RATING_COLS}')

def add_rating_aggregates(df):
    out = df.copy()
    out['rating_mean'] = out[RATING_COLS].mean(axis=1)
    out['rating_low_count'] = (out[RATING_COLS] <= 2).sum(axis=1)
    return out

features40 = add_rating_aggregates(features)
print(f'Exp 40 frame: {features40.shape[1]} columns ({features.shape[1]} raw + 2 aggregates); '
      f'counts/TE still on the original {len(TE_COLS_ALL)}')

print(f'Exp 40: counts + all-{len(TE_COLS_ALL)} TE + raw rating aggregates + stochastic, 3 seeds x {{63, 127}} leaves')
LEAF_CONFIGS = [63, 127]
oof_runs40, iters40 = {}, {}
for leaves in LEAF_CONFIGS:
    for seed in [42, 43, 44]:
        o, a, it = oof_lgbm(features40, target, f'E40-l{leaves}-s{seed}', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, **STOCH)
        oof_runs40[(leaves, seed)] = o
        iters40[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof40 = np.mean(list(oof_runs40.values()), axis=0)
aucs40 = fold_aucs(oof40, target)
print(f'Exp 40 (averaged) fold AUCs: {[round(a, 5) for a in aucs40]}')
print(f'Exp 40 mean OOF AUC: {np.mean(aucs40):.6f} +/- {np.std(aucs40):.5f} | pooled: {roc_auc_score(target, oof40):.6f}')

13 rating columns identified by exclusion: ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
Exp 40 frame: 23 columns (21 raw + 2 aggregates); counts/TE still on the original 21
Exp 40: counts + all-21 TE + raw rating aggregates + stochastic, 3 seeds x {63, 127} leaves


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E40-l63-s42 fold 1: AUC 0.95983 (best_iteration=255)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E40-l63-s42 fold 2: AUC 0.95842 (best_iteration=193)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E40-l127-s44 fold 4: AUC 0.95974 (best_iteration=189)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E40-l127-s44 fold 5: AUC 0.95983 (best_iteration=149)
  leaves=127 seed 44 mean AUC: 0.95974
Exp 40 (averaged) fold AUCs: [0.96009, 0.95884, 0.96063, 0.96006, 0.9601]


Exp 40 mean OOF AUC: 0.959945 +/- 0.00059 | pooled: 0.959939


## Exp 41: seed-averaged RandomForest on counts + leak-free TE

Exp 37 exactly, at seeds 42/43/44, OOF averaged per row - one change (seed averaging) from the
single-seed RF. Measures the forest family's own averaging gain at this fixed config with three seeds.
It sizes the single-seed-vs-3-seed effect carried as a caveat since round 13; it does not
exclude other forms of averaging asymmetry.

In [5]:
print('Exp 41: RandomForest 400/leaf5 on counts + leak-free TE, seeds 42/43/44, averaged')
oof_seeds41 = {}
for seed in [42, 43, 44]:
    o, a = oof_forest(features, target, f'E41-s{seed}', 'rf', seed=seed)
    oof_seeds41[seed] = o
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof41 = np.mean([oof_seeds41[s] for s in [42, 43, 44]], axis=0)
aucs41 = fold_aucs(oof41, target)
print(f'Exp 41 (averaged) fold AUCs: {[round(a, 5) for a in aucs41]}')
print(f'Exp 41 mean OOF AUC: {np.mean(aucs41):.6f} +/- {np.std(aucs41):.5f} | pooled: {roc_auc_score(target, oof41):.6f}')
m_seed, s_seed, lab_seed = paired_verdict(aucs41, E37_FOLDS_R13)
print(f'  averaging gain vs single-seed Exp 37 folds: {m_seed:+.6f} (spread {s_seed:.5f}) - {lab_seed}')

Exp 41: RandomForest 400/leaf5 on counts + leak-free TE, seeds 42/43/44, averaged


  E41-s42 fold 1: AUC 0.95756


  E41-s42 fold 2: AUC 0.95691


  E41-s42 fold 3: AUC 0.95824


  E41-s42 fold 4: AUC 0.95789


...

  E41-s44 fold 2: AUC 0.95687


  E41-s44 fold 3: AUC 0.95831


  E41-s44 fold 4: AUC 0.95797


  E41-s44 fold 5: AUC 0.95761
  seed 44 mean AUC: 0.95765
Exp 41 (averaged) fold AUCs: [0.95761, 0.95698, 0.9584, 0.95804, 0.95764]


Exp 41 mean OOF AUC: 0.957732 +/- 0.00047 | pooled: 0.957725
  averaging gain vs single-seed Exp 37 folds: +0.000104 (spread 0.00004) - WORKING SIGNAL: better (descriptive, not proof)


## Verdicts and refit guard

Each candidate pairs against the in-run incumbent, the Exp 29 folds, and the Exp 36 bar.
Refit fires only on a WORKING SIGNAL better vs the bar; on a tie or loss, Exp 36 and
submission_round12.csv stand. Bounded to the tested configurations.

In [6]:
results = {}
for tag, aucs, desc in [('40', aucs40, 'Exp 40: E36 scheme + raw rating aggregates'),
                        ('41', aucs41, 'Exp 41: seed-averaged RF on counts+TE')]:
    print(desc)
    m_, s_, lab = paired_verdict(aucs, INC_FOLDS)
    print(f'  vs incumbent (in-run):          mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_, s_, lab = paired_verdict(aucs, E29_FOLDS_R9)
    print(f'  vs Exp 29 folds (round 9):      mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_bar, s_bar, lab_bar = paired_verdict(aucs, E36_FOLDS_R12)
    print(f'  vs Exp 36 folds (round 12 bar): mean delta {m_bar:+.5f} (spread {s_bar:.5f}) - {lab_bar}')
    results[tag] = (m_bar, lab_bar, float(np.mean(aucs)))

winners = [k for k, (m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: results[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 36 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 36 bar under the paired band. Exp 36 remains the '
          'preferred candidate; submission_round12.csv is untouched.')

Exp 40: E36 scheme + raw rating aggregates
  vs incumbent (in-run):          mean delta +0.00147 (spread 0.00016) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 29 folds (round 9):      mean delta +0.00062 (spread 0.00018) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 36 folds (round 12 bar): mean delta -0.00005 (spread 0.00002) - WORKING SIGNAL: worse (descriptive, not proof)
Exp 41: seed-averaged RF on counts+TE
  vs incumbent (in-run):          mean delta -0.00074 (spread 0.00029) - WORKING SIGNAL: worse (descriptive, not proof)
  vs Exp 29 folds (round 9):      mean delta -0.00160 (spread 0.00035) - WORKING SIGNAL: worse (descriptive, not proof)
  vs Exp 36 folds (round 12 bar): mean delta -0.00226 (spread 0.00026) - WORKING SIGNAL: worse (descriptive, not proof)

REFIT GUARD: nothing cleared the Exp 36 bar under the paired band. Exp 36 remains the preferred candidate; submission_round12.csv is untouched.


## Conditional refit (only if the guard fired)

Exp 40 branch: round-12 refit semantics on the aggregate-augmented frame (full-train counts +
cross-fit TE on the original columns, aggregates raw, 3 seeds x {63, 127} leaves, n_estimators
basis from the in-run seed-42 fold-mean best_iterations). Exp 41 branch: round-13 forest refit
semantics, seed-averaged (3 full-train fits).

In [7]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win == '40':
        basis = int(round(np.mean([np.mean(iters40[(l, 42)]) for l in LEAF_CONFIGS])))
        feats_full, feats_test = add_rating_aggregates(features), add_rating_aggregates(test_features)
        tr_c, (te_c,) = add_counts(feats_full, [feats_test])
        te_full = te_crossfit(tr_c, target, TE_COLS_ALL)
        te_test = te_apply(features, target, test_features, TE_COLS_ALL)
        X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
        X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
        X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
        preds = []
        for leaves in LEAF_CONFIGS:
            for seed in [42, 43, 44]:
                cfg = dict(learning_rate=0.05, num_leaves=leaves, n_estimators=basis,
                           n_jobs=-1, random_state=seed, verbose=-1)
                cfg.update(STOCH)
                model = lgb.LGBMClassifier(**cfg)
                model.fit(X_full, target)
                preds.append(model.predict_proba(X_test)[:, 1])
        p = np.mean(preds, axis=0)
        scheme = f'Exp 40: E36 scheme + raw rating aggregates, 6 fits, basis {basis}'
    else:  # win == '41'
        X_full_f, X_test_f = forest_prep(features, target, test_features)
        preds = []
        for seed in [42, 43, 44]:
            rf = RandomForestClassifier(**{**RF_PARAMS, 'random_state': seed})
            rf.fit(X_full_f, target)
            preds.append(rf.predict_proba(X_test_f)[:, 1])
        p = np.mean(preds, axis=0)
        scheme = 'Exp 41: seed-averaged RandomForest 400/leaf5 on counts + TE, 3 fits'
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': p})
    sub.to_csv('submission_round16.csv', index=False)
    print(f'submission_round16.csv written: {len(sub):,} rows | {scheme}. '
          f'Local file only - NOT submitted; Andrea submits.')

No submission file written this round.


In [8]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round16_oof.npz',
         incumbent=oof_inc, exp40=oof40, exp41=oof41, target=target.to_numpy())
meta = {
    'round': 16,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'exp40': {'scheme': 'Exp 36 + raw rating aggregates (rating_mean, rating_low_count); counts/TE on original columns only'},
        'exp41': {'scheme': 'Exp 37 with 3-seed averaging (42/43/44), RF 400/leaf5'},
        'rating_cols': RATING_COLS,
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp40': aucs40, 'exp41': aucs41},
    'guard': {'winner': win, 'bar': 'E36_FOLDS_R12'},
}
(cache_dir / 'round16_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round16_oof.npz + round16_oof.json')

cached oof_cache/round16_oof.npz + round16_oof.json


## What happens next

- If Exp 40 wins: the aggregate block earns its place on this representation (bounded to these
  two aggregates) - submission_round16.csv is the new local candidate.
- If Exp 41 wins: the averaged forest earns its place (bounded to this config) - same.
- If both null: these two configurations are measured without a detected increment, bounded
  to themselves - not evidence that existing-data levers as a whole are exhausted. The
  remaining documented directions on existing data are narrower (parameter-diversity axes,
  other feature families), and each is a fresh adaptive-selection bet on the same OOF. The
  state of campaign goes to Andrea plainly, and external data (host-linked original
  dataset, CC0 on its face, provenance caveats logged) stays available as his explicit
  choice on its own merits - never an automatic step, never framed as the only road left.